# 01 — Data Understanding & Exploratory Data Analysis

First pass at understanding the two raw datasets before any cleaning, transformation, or analysis begins.

**Ground rules for this notebook:**
- No data cleaning
- No visualizations
- No conclusions
- The property tax file (~423 MB) is sampled at 1,000 rows only

---

## 1. Project Context

This project examines the relationship between **housing supply** and **assessed property values** in the City of Vancouver.

We are working with two raw datasets:

| Dataset | File | Role |
|---|---|---|
| Property Tax Report | `property_tax_report_raw.csv` | Assessed land and improvement values |
| Issued Building Permits | `issued_building_permits_raw.csv` | Permit activity as a housing supply signal |

**Why sample first?**  
The property tax file is approximately 423 MB. Loading it in full during exploration would be slow and memory-intensive. We load 1,000 rows to understand structure and data quality, then scale up only when needed.

## 2. Import Libraries

We use only the Python standard library (`os`, `csv`) and `pandas`. No additional packages are needed at this stage.

In [ ]:
import os       # file path operations and existence checks
import csv      # delimiter sniffing without loading the full file
import pandas as pd  # tabular data loading and inspection

# Show up to 50 columns in notebook output (default pandas limit is 20)
pd.set_option('display.max_columns', 50)

print('pandas version:', pd.__version__)

> **What to check:** `pandas` should import without errors. If you see a `ModuleNotFoundError`, activate your virtual environment and run `pip install -r requirements.txt` from the project root, then restart the kernel.

## 3. Define File Paths

All raw files live in `data/raw/`. Paths are defined once here so they are easy to update if files move. We also verify that both files exist and display their sizes before attempting to load anything.

In [ ]:
# Navigate up one level from notebooks/ to the project root
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

PROPERTY_TAX_PATH = os.path.join(BASE_DIR, 'data', 'raw', 'property_tax_report_raw.csv')
PERMITS_PATH      = os.path.join(BASE_DIR, 'data', 'raw', 'issued_building_permits_raw.csv')

# Confirm both files exist before proceeding
for label, path in [('Property Tax', PROPERTY_TAX_PATH), ('Permits', PERMITS_PATH)]:
    exists = os.path.isfile(path)
    size   = os.path.getsize(path) if exists else 0
    status = 'FOUND' if exists else 'MISSING'
    print(f'{label:15s} : {status} — {size:,} bytes')
    print(f'               {path}')

> **What to check:** Both files should show `FOUND`. Expected sizes: ~443,785,841 bytes for the property tax file and ~56,671,596 bytes for the permits file. If a file shows `MISSING`, confirm it was downloaded and placed in `data/raw/` with the exact filenames defined above.

## 4. Detect CSV Delimiter

Most CSV files use a comma (`,`), but some use semicolons (`;`), tabs (`\t`), or pipes (`|`). Python's built-in `csv.Sniffer` reads a small chunk of the file and infers the delimiter automatically.

Detecting this upfront prevents a common silent failure where every row loads as a single column because the wrong separator was assumed.

In [ ]:
def detect_delimiter(filepath, num_bytes=8192):
    '''
    Read a small chunk of the file and sniff its CSV delimiter.
    Falls back to comma if sniffing is inconclusive.
    '''
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        sample = f.read(num_bytes)
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=',;\t|')
        return dialect.delimiter
    except csv.Error:
        return ','  # safe default

prop_delim    = detect_delimiter(PROPERTY_TAX_PATH)
permits_delim = detect_delimiter(PERMITS_PATH)

print(f'Property Tax delimiter : {repr(prop_delim)}')
print(f'Permits delimiter      : {repr(permits_delim)}')

> **What to check:** Both should return `','`. If you see `';'` or `'\t'`, the file uses a non-standard separator — the loader below will still work correctly since it uses the detected delimiter. If the output shows `','` but data later appears in a single column, open the file in a text editor to verify and override `sep` manually.

## 5. Load a Small Sample (1,000 Rows)

We load the first 1,000 rows of each file using `nrows=1000`. This is enough to inspect structure and data quality without reading hundreds of megabytes into memory.

`low_memory=False` tells pandas to infer column types using the full sample rather than guessing in chunks, which avoids mixed-type warnings on large files.

In [ ]:
# Load first 1,000 rows only — do not remove the nrows limit during EDA
prop_sample = pd.read_csv(
    PROPERTY_TAX_PATH,
    nrows=1000,
    sep=prop_delim,
    low_memory=False,
    encoding='utf-8',
    encoding_errors='replace',  # replace unreadable characters instead of raising an error
)

permits_sample = pd.read_csv(
    PERMITS_PATH,
    nrows=1000,
    sep=permits_delim,
    low_memory=False,
    encoding='utf-8',
    encoding_errors='replace',
)

print('Samples loaded successfully.')

> **What to check:** You should see `Samples loaded successfully.` If you get a `FileNotFoundError`, re-run the file path cell above. A persistent `UnicodeDecodeError` can be resolved by switching to `encoding='latin-1'`. If a file loads but all data appears in one column, the sniffed delimiter was incorrect — override `sep` with the correct character.

## 6. Shape of Each Sample

`.shape` returns `(rows, columns)`. Since we set `nrows=1000`, the row count will be 1,000 (unless the file has fewer rows). The **column count** is what tells us whether the delimiter was parsed correctly.

In [ ]:
print(f'Property Tax sample shape  : {prop_sample.shape}   → (rows, columns)')
print(f'Permits sample shape       : {permits_sample.shape}   → (rows, columns)')

> **What to check:** The column count should be greater than 3–4. A result of `(1000, 1)` means the file was not parsed correctly — all data ended up in a single column, which usually means the wrong delimiter was detected.

## 7. Column Names

Listing all column names gives us a map of every available field. We will use this to populate `docs/data_dictionary.md` and decide which columns are relevant to the analysis.

In [ ]:
print('=== Property Tax Columns ===')
for i, col in enumerate(prop_sample.columns):
    print(f'  [{i:02d}] {col}')

print()

print('=== Permits Columns ===')
for i, col in enumerate(permits_sample.columns):
    print(f'  [{i:02d}] {col}')

> **What to look for:**
> - **Property Tax:** fields for assessed value, land value, improvement value, a tax or assessment year, and a property identifier (parcel ID or PID). Also note any address, neighbourhood, or zoning fields that could support geographic grouping.
> - **Permits:** fields for permit type, issue date, project value, address, and type of work (new construction vs. renovation vs. demolition). A date field and a type/category field are both essential for the time-trend analysis.
> - Flag any column names that are unclear, unexpectedly absent, or appear duplicated — note them for follow-up.

## 8. First 5 Rows

Viewing actual data values helps spot formatting issues before any analysis: currency symbols embedded in numeric fields, inconsistent date formats, extra whitespace, or placeholder values standing in for nulls.

In [ ]:
print('=== Property Tax — First 5 Rows ===')
display(prop_sample.head())

print()
print('=== Permits — First 5 Rows ===')
display(permits_sample.head())

> **What to look for:**
> - Are value/amount columns stored as numbers, or as strings containing `$` or `,`?
> - Are date columns formatted consistently (e.g., `YYYY-MM-DD`, `DD/MM/YYYY`, `Month DD, YYYY`)?
> - Do any fields contain free-text or mixed content that will need parsing later?
> - Are there visible placeholder values — `0`, `N/A`, `UNKNOWN`, `-`, or empty strings — that may represent missing data but won't be caught by `isnull()`?

## 9. Data Types

`pandas` assigns a `dtype` to each column when loading. The main types to watch for:

| dtype | Meaning |
|---|---|
| `int64` | Integer number |
| `float64` | Decimal number |
| `object` | String or mixed content |
| `bool` | True / False |

Columns you expect to be numeric but show `object` likely contain formatting characters (like `$` or `,`) and will need cleaning before any arithmetic can be done on them.

In [ ]:
print('=== Property Tax — Data Types ===')
print(prop_sample.dtypes.to_string())

print()

print('=== Permits — Data Types ===')
print(permits_sample.dtypes.to_string())

> **What to look for:**
> - **Value columns** (assessed value, permit value) should ideally be `float64` or `int64`. If they show `object`, they contain non-numeric characters — plan to strip and convert during cleaning.
> - **Date columns** will almost always load as `object`. These will need `pd.to_datetime()` conversion in the cleaning step.
> - **ID or code columns** may load as `int64` even if they shouldn't be treated as numbers (e.g., a parcel ID that starts with leading zeros). Flag these — they may need to be cast to `str`.

## 10. Missing Values (Sample)

We count null values per column across the 1,000-row sample. This is not a definitive view of the full dataset, but it gives an early signal of which fields have data quality gaps.

**Important:** `isnull()` only catches `NaN` / `None`. It will *not* catch empty strings, zeros used as placeholders, or text values like `N/A` — those require manual inspection of the raw values.

In [ ]:
def missing_summary(df, label):
    '''Print a summary of columns with missing values, sorted by count descending.'''
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(1)
    summary = pd.DataFrame({'missing_count': missing, 'missing_pct': pct})
    summary = summary[summary['missing_count'] > 0].sort_values('missing_count', ascending=False)

    if summary.empty:
        print(f'{label}: no missing values detected in sample ({len(df)} rows)')
    else:
        print(f'{label} — columns with missing values ({len(df)} rows sampled):')
        display(summary)
    print()

missing_summary(prop_sample,    'Property Tax')
missing_summary(permits_sample, 'Permits')

> **What to look for:**
> - Columns with a high missing rate (>50%) may be optional fields, sparsely populated categories, or legacy columns — investigate before deciding to drop them.
> - Missing values in **critical fields** (assessed value, permit date, address) will need an explicit resolution strategy in the cleaning notebook. Do not drop rows yet.
> - If critical columns show 0% missing, check whether zeros or placeholder strings are masking the true null rate — `df['column'].value_counts(dropna=False)` is useful for this.

## 11. Initial Questions to Investigate

Questions raised by this first inspection. These drive the cleaning and analysis work in subsequent notebooks. Move answered questions to `docs/methodology.md`.

### Property Tax Dataset
- What years does this dataset cover? Is there a tax year or assessment year column?
- Is each row one property for one year, or does one row span multiple years?
- What is the unique identifier for a property — is there a parcel ID (PID)?
- Are assessed values already numeric, or do they need parsing (e.g., currency symbols or commas)?
- Does the dataset include multiple property classes (residential, commercial, industrial)? Do we need to filter to residential only?

### Issued Building Permits Dataset
- What date range does this dataset cover — does it span 2019–2024 as expected?
- Is there a field that distinguishes new construction from renovations or demolitions?
- What geographic granularity is available — full address, neighbourhood, local planning area?
- Is there a field for the number of dwelling units permitted?
- Can permits be linked to the property tax dataset by address or parcel identifier?

### Cross-Dataset
- Do the two datasets share a geographic key (address, parcel ID, neighbourhood name) that could support a join?
- Are there time-alignment considerations — e.g., a permit issued in year X may not appear in the assessment roll until year X+1 or X+2?

---

*Update these questions as analysis progresses. Move resolved questions to `docs/methodology.md`.*